# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yara-08/ML-FlyRank-Internship2/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import duckdb
from google.colab import userdata

intern_token = userdata.get("intern_token")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{intern_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [18]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [19]:
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [20]:
march_data = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""")

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



---



> **Unit of analysis:** One row represents the daily performance of a single pseudonymized content item for one pseudonymized client on a specific report date.

> **Time window:** For this assignment, I use data from March 2026 (month = '2026-03'), which is a mid-panel month recommended by the internship. This avoids using the final month (June 2026), which is reserved as a future evaluation period. My analysis uses only information available during this month to support refresh opportunity scoring.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()


con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*




---



**Features (used by the model)**

* **gsc_impressions** – Measures how often a page appears in Google Search and is available before making a refresh decision.
* **gsc_clicks** – Measures search traffic already received and is known at the decision time.
* **gsc_avg_position** – Indicates the page's average search ranking before any refresh.
* **ga4_sessions** – Represents website traffic recorded before the refresh decision.
* **ga4_engaged_sessions** – Measures user engagement before the refresh and may help identify pages with improvement potential.

**Label / Proxy**

* **Refresh opportunity score (proxy)** – The target score that estimates which content items should be prioritized for refresh. It is the value the model is intended to predict and must never be used as an input feature.

**Context**

* **report_date** – Identifies when the observations were recorded and is used for filtering and defining the analysis window.
* **client_hash_id** – Identifies the pseudonymized client for grouping and analysis only.
* **content_hash_id** – Identifies the pseudonymized content item for grouping and joins.
* **month** – Used to select the March 2026 partition and improve query efficiency.

**Excluded**

* **gsc_data_available** and **ga4_data_available** – Used only to filter rows with valid data. They are not predictive features and should not be learned by the model.
* Any future or label-derived information is excluded because it would introduce data leakage and produce unrealistically optimistic model performance.


In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions,
    gsc_data_available,
    ga4_data_available,
    month
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,gsc_data_available,ga4_data_available,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,True,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,True,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,True,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,True,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,True,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



---


> I verified the data contract with three queries. First, I confirmed that each row is uniquely identified by the combination of `report_date`, `client_hash_id`, and `content_hash_id`, as no duplicate records were found. Second, I verified that my analysis window covers **March 2026**, from **2026-03-01** to **2026-03-31**, containing **9,841,378** rows. Finally, I filtered the data using `gsc_data_available IS TRUE` and `ga4_data_available IS TRUE`, leaving **364,347** rows with valid Search Console and Google Analytics data. These verified rows will be used for feature construction to avoid using missing or zero-filled observations.



In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,364347


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*





---




> This dataset is useful for supporting refresh decisions, but it cannot prove that refreshing a page will cause its future performance to improve. The data is observational rather than experimental, so it can identify patterns but not causal relationships. In addition, not every record contains both valid Google Search Console and Google Analytics data. For March 2026, the dataset contains **9,841,378** rows, but only **364,347** rows have both `gsc_data_available` and `ga4_data_available` set to `TRUE`. As a result, only a subset of the data is suitable for feature construction, and the results should be interpreted as decision support rather than causal proof.



In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
             AND ga4_data_available IS TRUE
            THEN 1
            ELSE 0
        END
    ) AS usable_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()

,total_rows,usable_rows
0,9841378,364347.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.